In [1]:
import os
os.environ['KAGGLE_USERNAME'] = 'token1'
os.environ['KAGGLE_KEY'] = 'KGAT_59b49d7ae2e470c8eb685104194a5317'



In [3]:
!pip install -q kaggle
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset
!unzip -q new-plant-diseases-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.70G/2.70G [00:34<00:00, 83.6MB/s]



In [2]:
from tensorflow.keras.applications import MobileNetV2


In [4]:
base_model = MobileNetV2(
    weights = 'imagenet',
    include_top = False,
    input_shape = (224,224,3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
base_model.trainable = False


In [6]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense,Dropout,GlobalAveragePooling2D
from keras import Sequential


In [7]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    directory = '/content/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (224,224),
    shuffle = True
)

Found 70295 files belonging to 38 classes.


In [8]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
    directory = '/content/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (224,224),
    shuffle = False
)

Found 17572 files belonging to 38 classes.


In [9]:
def process(image,label):
  image = tf.cast(image/255. , tf.float32)
  return image,label


train_dataset = train_dataset.map(process)
test_dataset = test_dataset.map(process)



In [10]:
model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dense(256,activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(128,activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(38,activation='softmax'))




In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,623,718 (10.01 MB)

 Trainable params: 365,734 (1.40 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [12]:
model.compile(loss='sparse_categorical_crossentropy',optimizer= 'adam',metrics=['accuracy'])

In [13]:
from tensorflow.keras.callbacks import EarlyStopping
callback = EarlyStopping(
    monitor='val_loss',
    min_delta=0.001,
    verbose=1,
    mode='auto',
    baseline=None,
    restore_best_weights=True,
)

In [14]:
history = model.fit(train_dataset,epochs=20,validation_data = test_dataset,callbacks=[callback])

Epoch 1/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 156s 61ms/step - accuracy: 0.8074 - loss: 0.6290 - val_accuracy: 0.9309 - val_loss: 0.2132
Epoch 2/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 101s 46ms/step - accuracy: 0.9044 - loss: 0.2979 - val_accuracy: 0.9422 - val_loss: 0.1741
Epoch 3/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 139s 45ms/step - accuracy: 0.9202 - loss: 0.2462 - val_accuracy: 0.9484 - val_loss: 0.1495
Epoch 4/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 96s 44ms/step - accuracy: 0.9294 - loss: 0.2189 - val_accuracy: 0.9556 - val_loss: 0.1327
Epoch 5/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 94s 43ms/step - accuracy: 0.9386 - loss: 0.1908 - val_accuracy: 0.9469 - val_loss: 0.1609
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 4.


In [15]:
pred = model.evaluate(test_dataset)

550/550 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.9556 - loss: 0.1327


In [16]:
prediction = model.predict(test_dataset)
prediction

550/550 ━━━━━━━━━━━━━━━━━━━━ 29s 44ms/step


array([[9.99907613e-01, 6.47865249e-07, 9.17505458e-05, ...,
        4.53485310e-18, 1.52876973e-17, 6.08281835e-16],
       [9.99896169e-01, 2.12024088e-06, 1.01604128e-04, ...,
        1.04475802e-13, 2.32854507e-13, 6.73839511e-13],
       [9.94303882e-01, 7.44102465e-04, 4.94637899e-03, ...,
        4.23394875e-10, 4.07595832e-11, 1.93948058e-08],
       ...,
       [3.29330351e-06, 2.03321946e-07, 5.69820349e-06, ...,
        1.22618826e-06, 1.02533079e-06, 9.96208429e-01],
       [1.34740631e-14, 1.64010976e-16, 1.71473875e-14, ...,
        1.56979020e-12, 1.79813421e-11, 9.99989867e-01],
       [2.07310704e-13, 1.47673979e-15, 2.77860336e-13, ...,
        1.13228024e-10, 8.63347838e-10, 9.96912599e-01]], dtype=float32)

In [17]:
import numpy as np
y_true = np.concatenate([y for x , y in test_dataset],axis=0)

In [18]:
from sklearn.metrics import confusion_matrix,classification_report
y_pred = np.argmax(prediction,axis=1)
print(classification_report(y_true,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.95      0.96       504
           1       0.96      1.00      0.98       497
           2       0.99      0.97      0.98       440
           3       0.99      0.98      0.99       502
           4       0.98      1.00      0.99       454
           5       0.99      1.00      0.99       421
           6       0.98      0.99      0.99       456
           7       0.98      0.85      0.91       410
           8       0.99      0.99      0.99       477
           9       0.88      0.99      0.93       477
          10       1.00      1.00      1.00       465
          11       0.99      0.92      0.95       472
          12       0.93      0.99      0.96       480
          13       0.99      1.00      0.99       430
          14       0.99      1.00      1.00       423
          15       0.99      1.00      0.99       503
          16       0.99      0.98      0.98       459
          17       0.99    

In [19]:
confusion_matrix(y_true,y_pred)

array([[480,  10,   0, ...,   0,   0,   0],
       [  0, 496,   0, ...,   0,   0,   0],
       [  5,   0, 427, ...,   0,   0,   0],
       ...,
       [  0,   0,   0, ..., 475,   2,   0],
       [  0,   0,   0, ...,   0, 440,   0],
       [  0,   0,   1, ...,   0,   0, 445]])

In [20]:
base_model.trainable = True

In [21]:
for layer in base_model.layers[:-20]:
  layer.trainable = False

In [22]:
 model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),metrics=['accuracy'],loss='sparse_categorical_crossentropy')

In [23]:
 history = model.fit(train_dataset,epochs=20,validation_data=test_dataset,callbacks=callback)

Epoch 1/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 132s 53ms/step - accuracy: 0.8739 - loss: 0.4447 - val_accuracy: 0.9539 - val_loss: 0.1449
Epoch 2/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 100s 46ms/step - accuracy: 0.9337 - loss: 0.2046 - val_accuracy: 0.9626 - val_loss: 0.1147
Epoch 3/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 103s 47ms/step - accuracy: 0.9510 - loss: 0.1531 - val_accuracy: 0.9673 - val_loss: 0.0975
Epoch 4/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 103s 47ms/step - accuracy: 0.9602 - loss: 0.1220 - val_accuracy: 0.9718 - val_loss: 0.0874
Epoch 5/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 122s 55ms/step - accuracy: 0.9658 - loss: 0.1062 - val_accuracy: 0.9744 - val_loss: 0.0758
Epoch 6/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 119s 45ms/step - accuracy: 0.9709 - loss: 0.0886 - val_accuracy: 0.9756 - val_loss: 0.0713
Epoch 7/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 101s 46ms/step - accuracy: 0.9758 - loss: 0.0765 - val_accuracy: 0.9781 - val_loss: 0.0670
Epoch 8/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 101s 46ms/step - accuracy: 

In [24]:
model.evaluate(test_dataset)

550/550 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.9830 - loss: 0.0532


[0.05316775664687157, 0.9830412268638611]

In [25]:
pred = model.predict(test_dataset)
pred

550/550 ━━━━━━━━━━━━━━━━━━━━ 29s 44ms/step


array([[1.0000000e+00, 1.1880256e-11, 1.6556852e-08, ..., 6.4050107e-26,
        1.3580438e-23, 1.0887122e-22],
       [9.9999607e-01, 1.4292614e-07, 3.7880764e-06, ..., 1.0691937e-17,
        1.1710638e-16, 2.6019287e-15],
       [9.9998558e-01, 1.3295901e-08, 1.4450655e-05, ..., 1.8054558e-16,
        4.7061643e-18, 5.2093787e-14],
       ...,
       [1.8149620e-12, 5.2456592e-16, 1.5559627e-11, ..., 2.8191045e-13,
        9.8733264e-14, 9.9999940e-01],
       [1.5138651e-21, 4.2228726e-26, 5.1861579e-21, ..., 4.4736985e-21,
        8.3772852e-20, 1.0000000e+00],
       [9.2231836e-17, 6.1519502e-21, 1.2690602e-16, ..., 3.4423919e-17,
        4.0790511e-15, 9.9999964e-01]], dtype=float32)

In [26]:
predicted = np.argmax(pred,axis=1)

In [27]:
print(classification_report(y_true,predicted))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98       504
           1       0.98      1.00      0.99       497
           2       0.99      1.00      1.00       440
           3       1.00      1.00      1.00       502
           4       1.00      1.00      1.00       454
           5       0.99      1.00      1.00       421
           6       1.00      1.00      1.00       456
           7       0.97      0.91      0.94       410
           8       1.00      1.00      1.00       477
           9       0.93      0.97      0.95       477
          10       1.00      1.00      1.00       465
          11       1.00      0.99      0.99       472
          12       0.99      1.00      0.99       480
          13       1.00      1.00      1.00       430
          14       1.00      1.00      1.00       423
          15       1.00      1.00      1.00       503
          16       0.99      0.99      0.99       459
          17       0.99    